# testing_adaptive_2.ipynb

Structured notebook for **single-run simulation** of nonlinear dynamical systems:

- Integration (solve_ivp / RK4)
- Phase portrait (2D)
- Lyapunov exponents vs time

Designed to be driven by a **StaticParamsConfig.json** export from the Streamlit app.


In [ ]:
# --- Imports + project root wiring ---
import json
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / "core").is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate repo root (missing core/)")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.solver import integrate_system, integrate_system_rk4
from core.lyapunov import (
    _integrate_chunk_ivp,
    _pack_augmented,
    _qr_accumulate,
    _unpack_augmented,
    finite_difference_jacobian,
)
from core.lorenz_system_rhs import lorenz_rhs
from core.rossler_system_rhs import rossler_rhs
from core.jacobians_fixed_systems import lorenz_jac, rossler_jac

from plotting.plotting import phase_portrait

print("OK: imports loaded.")


## 1) Load config (StaticParamsConfig.json)

If you exported a config from the app, place it next to this notebook as `StaticParamsConfig.json`
(or change the path below).


In [ ]:
# --- Load StaticParamsConfig.json (if present). Otherwise use a default config. ---
CONFIG_PATH = Path("StaticParamsConfig.json")

def load_config(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

if CONFIG_PATH.exists():
    cfg = load_config(CONFIG_PATH)
    print(f"Loaded config: {CONFIG_PATH.resolve()}")
else:
    # Default fallback config (Lorenz)
    cfg = {
        "schema_version": "1.0",
        "system": {
            "system_key": "lorenz",
            "var_names": ["x", "y", "z"],
            "eq_lines": [],
            "params": {"sigma": 10.0, "rho": 28.0, "beta": 8.0 / 3.0},
        },
        "integration": {
            "t0": 0.0, "tf": 50.0, "dt": 0.01,
            "y0": [1.0, 1.0, 1.0],
            "solver_kind": "ivp",
            "solve_options": {"rtol": 3e-4, "atol": 1e-6},
        },
        "postprocess": {"transient_steps": 0},
        "plots": {
            "plot_mode": "2D phase plane",
            "phase_axes": {"x_idx": 0, "y_idx": 1, "z_idx": 2},
        },
        "lyapunov": {
            "enabled": True,
            "settings": {
                "qr_interval": 0.1,
                "jacobian": "analytic",
                "fd_eps": 1e-8,
                "t_transient": 0.0,
                "t_measure": 50.0,
                "transient_steps": 0,
                "transient_fraction": 0.0,
            },
        },
    }
    print("No StaticParamsConfig.json found — using default Lorenz config.")

cfg


## 2) Build RHS (built-in or custom)

Supported `system_key`:
- `"lorenz"`
- `"rossler"`
- `"custom"` (uses SymPy expressions in `eq_lines`)

For `custom`, config must include `var_names` and `eq_lines` (one equation per variable).


In [ ]:
SAFE_FUNCS = {
    "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "exp": sp.exp, "log": sp.log, "sqrt": sp.sqrt,
    "abs": sp.Abs,
}


def parse_params_text(text: str) -> dict:
    params = {}
    for line in (text or "").splitlines():
        line = line.strip()
        if not line:
            continue
        if "=" not in line:
            raise ValueError(f"Parameter line must be name=value. Got: {line!r}")
        name, val = line.split("=", 1)
        name = name.strip()
        val = val.strip()
        if name.lower() == "t":
            raise ValueError("Parameter name 't' is reserved.")
        params[name] = float(val)
    return params


def build_custom_rhs_and_jac(var_names, eq_lines, params, use_jac=True):
    t_sym = sp.Symbol("t")
    var_syms = sp.symbols(var_names)
    param_syms = {k: sp.Symbol(k) for k in (params or {}).keys()}

    locals_dict = {
        **SAFE_FUNCS,
        "t": t_sym,
        **{name: sym for name, sym in zip(var_names, var_syms)},
        **param_syms,
    }

    exprs = []
    for line in eq_lines:
        exprs.append(sp.sympify(line, locals=locals_dict))

    args = [t_sym] + list(var_syms) + [param_syms[k] for k in params.keys()]
    f_rhs = sp.lambdify(args, exprs, modules=["numpy"])
    J = sp.Matrix(exprs).jacobian(var_syms)
    f_jac = sp.lambdify(args, J, modules=["numpy"]) if use_jac else None

    param_values = [float(params[k]) for k in params.keys()]

    def rhs(t, y):
        vals = [float(t)] + list(np.asarray(y, dtype=float)) + param_values
        out = f_rhs(*vals)
        return np.array(out, dtype=float)

    def jac(t, y):
        if f_jac is None:
            return None
        vals = [float(t)] + list(np.asarray(y, dtype=float)) + param_values
        return np.array(f_jac(*vals), dtype=float)

    return rhs, jac


def build_rhs_and_jac(cfg: dict):
    sys_cfg = cfg.get("system", {})
    key = str(sys_cfg.get("system_key", "")).lower().strip()
    params = sys_cfg.get("params") or {}
    if not params and sys_cfg.get("params_text"):
        params = parse_params_text(sys_cfg.get("params_text"))
    params = {str(k): float(v) for k, v in (params or {}).items()}

    if key == "lorenz":
        return (
            lambda t, y: lorenz_rhs(t, y, **params),
            lambda t, y: lorenz_jac(t, y, **params),
        )
    if key == "rossler":
        return (
            lambda t, y: rossler_rhs(t, y, **params),
            lambda t, y: rossler_jac(t, y, **params),
        )
    if key == "custom":
        var_names = sys_cfg.get("var_names") or []
        eq_lines = sys_cfg.get("eq_lines") or []
        if len(var_names) == 0 or len(eq_lines) == 0:
            raise ValueError("Custom system requires var_names and eq_lines in config.")
        if len(eq_lines) != len(var_names):
            raise ValueError("Custom system: eq_lines must match var_names length.")
        auto_jac = bool(sys_cfg.get("auto_jacobian", False))
        use_jac = bool(sys_cfg.get("use_jacobian", False))
        rhs, jac = build_custom_rhs_and_jac(var_names, eq_lines, params, use_jac=auto_jac and use_jac)
        return rhs, jac

    raise ValueError(f"Unknown system_key: {key!r}")


rhs, jac = build_rhs_and_jac(cfg)
print("OK: RHS built.")


## 3) Integrate the system


In [ ]:
integ = cfg.get("integration", {})
t0 = float(integ.get("t0", 0.0))
tf = float(integ.get("tf", 50.0))
dt = float(integ.get("dt", 0.01))
y0 = np.asarray(integ.get("y0", [1.0, 1.0, 1.0]), dtype=float)

solver_kind = str(integ.get("solver_kind", "ivp")).lower().strip()
solve_opts = integ.get("solve_options") or {}

if solver_kind == "rk4":
    sol = integrate_system_rk4(rhs, (t0, tf), y0, t_step=dt)
else:
    sol = integrate_system(rhs, (t0, tf), y0, t_step=dt, **solve_opts)

print("success:", sol.success)
print("message:", sol.message)
print("t.shape:", sol.t.shape, "y.shape:", sol.y.shape)

t = sol.t
Y = sol.y  # shape (n_states, n_points)


## 4) Phase portrait (2D)

We use `phase_portrait` from `plotting/plotting.py`. If the config plot mode is 3D,
we still plot a 2D projection using `x_idx` and `y_idx`.


In [ ]:
plots_cfg = cfg.get("plots") or {}
axes_cfg = plots_cfg.get("phase_axes") or {"x_idx": 0, "y_idx": 1}
xi = int(axes_cfg.get("x_idx", 0))
yi = int(axes_cfg.get("y_idx", 1))

transient_steps = int((cfg.get("postprocess") or {}).get("transient_steps", 0))
transient_steps = max(0, min(transient_steps, Y.shape[1] - 1))

var_names = (cfg.get("system") or {}).get("var_names") or [f"y{i+1}" for i in range(Y.shape[0])]

phase_portrait(
    Y,
    x_index=xi,
    y_index=yi,
    transient_steps=transient_steps,
    xlabel=var_names[xi],
    ylabel=var_names[yi],
    title=f"Phase portrait ({var_names[yi]} vs {var_names[xi]})",
)


## 5) Lyapunov exponents vs time

This computes a *time series* of Lyapunov estimates using QR re-orthonormalization.
We use the full post-transient window.


In [ ]:
def compute_lyapunov_time_series(
    rhs,
    x0,
    t0,
    tf,
    dt,
    t_transient,
    qr_interval,
    solve_options=None,
    jac=None,
    fd_eps=1e-8,
):
    if dt <= 0.0:
        raise ValueError("dt must be > 0.")
    if qr_interval <= 0.0:
        raise ValueError("qr_interval must be > 0.")

    x = np.asarray(x0, dtype=float).copy()
    n = x.size
    if n < 1:
        raise ValueError("x0 must have at least one component.")

    if jac is None:
        def jac_fn(tt, xx):
            return finite_difference_jacobian(rhs, tt, xx, eps=fd_eps)
    else:
        jac_fn = jac

    Q = np.eye(n, dtype=float)

    def rhs_aug(tt, y_aug):
        xx, QQ = _unpack_augmented(y_aug, n)
        dx = rhs(tt, xx)
        J = jac_fn(tt, xx)
        dQ = J @ QQ
        return _pack_augmented(dx, dQ)

    qr_every_steps = max(1, int(round(float(qr_interval) / float(dt))))
    chunk_dt = float(qr_every_steps) * float(dt)

    t = float(t0)
    y_aug = _pack_augmented(x, Q)

    n_full = int(np.floor(t_transient / chunk_dt))
    rem = float(t_transient) - n_full * chunk_dt
    for _ in range(n_full):
        y_aug = _integrate_chunk_ivp(rhs_aug, t, y_aug, t + chunk_dt, solve_options=solve_options)
        t += chunk_dt
        x, Q = _unpack_augmented(y_aug, n)
        Q, _ = np.linalg.qr(Q)
        y_aug = _pack_augmented(x, Q)
    if rem > 1e-15:
        y_aug = _integrate_chunk_ivp(rhs_aug, t, y_aug, t + rem, solve_options=solve_options)
        t += rem
        x, Q = _unpack_augmented(y_aug, n)
        Q, _ = np.linalg.qr(Q)
        y_aug = _pack_augmented(x, Q)

    t_measure = float(tf) - float(t0) - float(t_transient)
    if t_measure <= 0.0:
        raise ValueError("No measurement time available for Lyapunov.")

    sums_log = np.zeros(n, dtype=float)
    times = []
    lambdas = []
    elapsed = 0.0

    n_full = int(np.floor(t_measure / chunk_dt))
    rem = float(t_measure) - n_full * chunk_dt
    for _ in range(n_full):
        y_aug = _integrate_chunk_ivp(rhs_aug, t, y_aug, t + chunk_dt, solve_options=solve_options)
        t += chunk_dt
        elapsed += chunk_dt
        x, Q = _unpack_augmented(y_aug, n)
        Q, sums_log = _qr_accumulate(Q, sums_log)
        times.append(float(t))
        lambdas.append(sums_log / float(elapsed))
        y_aug = _pack_augmented(x, Q)
    if rem > 1e-15:
        y_aug = _integrate_chunk_ivp(rhs_aug, t, y_aug, t + rem, solve_options=solve_options)
        t += rem
        elapsed += rem
        x, Q = _unpack_augmented(y_aug, n)
        Q, sums_log = _qr_accumulate(Q, sums_log)
        times.append(float(t))
        lambdas.append(sums_log / float(elapsed))

    if not times:
        raise RuntimeError("No QR steps performed. Increase measurement time or reduce qr_interval.")

    return np.array(times, dtype=float), np.vstack(lambdas)


lyap_cfg = cfg.get("lyapunov") or {}
if not bool(lyap_cfg.get("enabled", True)):
    print("Lyapunov disabled in config.")
else:
    lyap_settings = lyap_cfg.get("settings") or {}
    qr_interval = float(lyap_settings.get("qr_interval", dt))
    if "qr_every_steps" in lyap_settings and qr_interval <= 0.0:
        qr_interval = float(lyap_settings.get("qr_every_steps", 1)) * float(dt)

    jac_mode = str(lyap_settings.get("jacobian", ""))
    jac_mode = jac_mode.lower().strip()
    fd_eps = float(lyap_settings.get("fd_eps", 1e-8))
    jac_to_use = jac if jac_mode == "analytic" else None

    post = cfg.get("postprocess") or {}
    lyap_t_transient = lyap_settings.get("t_transient", None)
    if lyap_t_transient is not None:
        t_transient = float(lyap_t_transient)
    else:
        lyap_transient_steps = lyap_settings.get("transient_steps", None)
        if lyap_transient_steps is not None:
            t_transient = float(int(lyap_transient_steps)) * dt
        else:
            t_transient = float(int(post.get("transient_steps", 0)) * dt)

    times, lambdas = compute_lyapunov_time_series(
        rhs=rhs,
        x0=Y[:, 0],
        t0=t0,
        tf=tf,
        dt=dt,
        t_transient=t_transient,
        qr_interval=qr_interval,
        solve_options=solve_opts,
        jac=jac_to_use,
        fd_eps=fd_eps,
    )

    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    fig.set_dpi(140)
    for i in range(lambdas.shape[1]):
        ax.plot(times, lambdas[:, i], linewidth=1.0, label=f"lambda{i}")
    ax.set_xlabel("t")
    ax.set_ylabel("Lyapunov exponents")
    ax.grid(True, linewidth=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.show()


## 6) QR interval plateau test (lambda1 vs time)

For each QR interval, plot λ₁(t) and check:
- whether it converges (plateau), and
- whether it stays the same when you increase the measurement window.


In [ ]:
if not bool(lyap_cfg.get("enabled", True)):
    print("Lyapunov disabled in config.")
else:
    qr_base = float(lyap_settings.get("qr_interval", dt))
    if qr_base <= 0.0:
        qr_base = float(dt)

    # Adjust this list as needed
    qr_intervals = [qr_base, 2.0 * qr_base, 5.0 * qr_base]
    qr_intervals = sorted({q for q in qr_intervals if q > 0.0})

    t_measure_base = float(tf) - float(t0) - float(t_transient)
    if t_measure_base <= 0.0:
        raise ValueError("No measurement time available for Lyapunov.")

    extend_factor = 2.0  # set to 1.0 to skip the longer window
    tf_ext = float(t0) + float(t_transient) + extend_factor * t_measure_base

    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    fig.set_dpi(140)

    for q in qr_intervals:
        times_base, lambdas_base = compute_lyapunov_time_series(
            rhs=rhs,
            x0=Y[:, 0],
            t0=t0,
            tf=tf,
            dt=dt,
            t_transient=t_transient,
            qr_interval=q,
            solve_options=solve_opts,
            jac=jac_to_use,
            fd_eps=fd_eps,
        )
        l1_base = np.max(lambdas_base, axis=1)
        ax.plot(times_base, l1_base, linewidth=1.0, label=f"QR={q:g} (base)")

        if extend_factor > 1.0:
            times_ext, lambdas_ext = compute_lyapunov_time_series(
                rhs=rhs,
                x0=Y[:, 0],
                t0=t0,
                tf=tf_ext,
                dt=dt,
                t_transient=t_transient,
                qr_interval=q,
                solve_options=solve_opts,
                jac=jac_to_use,
                fd_eps=fd_eps,
            )
            l1_ext = np.max(lambdas_ext, axis=1)
            ax.plot(times_ext, l1_ext, linewidth=1.0, linestyle="--", label=f"QR={q:g} (x{extend_factor:g})")

    ax.set_xlabel("t")
    ax.set_ylabel("lambda1(t)")
    ax.grid(True, linewidth=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.show()


## 7) Tolerances vs Lyapunov max value

We sweep absolute tolerance and set relative tolerance so that:
`rtol * y_scale == atol`, where `y_scale = max(|Y|)`.


In [ ]:
if not bool(lyap_cfg.get("enabled", True)):
    print("Lyapunov disabled in config.")
else:
    atols = [1e-6, 1e-7, 1e-8, 1e-9]
    y_scale = float(np.max(np.abs(Y)))
    y_scale = max(y_scale, 1.0)

    max_vals = []
    rtols = []

    for atol in atols:
        rtol = float(atol) / y_scale
        rtols.append(rtol)
        opts = dict(solve_opts)
        opts.update({"rtol": float(rtol), "atol": float(atol)})

        times_t, lambdas_t = compute_lyapunov_time_series(
            rhs=rhs,
            x0=Y[:, 0],
            t0=t0,
            tf=tf,
            dt=dt,
            t_transient=t_transient,
            qr_interval=qr_interval,
            solve_options=opts,
            jac=jac_to_use,
            fd_eps=fd_eps,
        )
        max_vals.append(float(np.max(lambdas_t[-1])))

    fig, ax = plt.subplots(figsize=(7.0, 4.0))
    fig.set_dpi(140)
    ax.semilogx(atols, max_vals, marker="o", linewidth=1.0)
    ax.set_xlabel("absolute tolerance (atol)")
    ax.set_ylabel("Max Lyapunov exponent")
    ax.grid(True, linewidth=0.3, which="both")
    plt.show()

    for atol, rtol, val in zip(atols, rtols, max_vals):
        print(f"atol={atol:.1e} rtol={rtol:.3e} max_lambda={val:.6f}")
